<a href="https://colab.research.google.com/github/team-telnyx/telnyx-code-examples/blob/main/cookbooks/inference/03_RAG_Telnyx_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with Telnyx Inference

In this notebook, you’ll build a simple Retrieval-Augmented Generation (RAG) workflow with Telnyx Inference.

You’ll:

1. Create a small knowledge base
2. Generate embeddings with Telnyx Inference
3. Retrieve the most relevant documents for a user question
4. Use a Telnyx-hosted chat model to answer using only the retrieved context

We’ll use the OpenAI Python SDK with Telnyx’s OpenAI-compatible endpoint. This notebook uses a simple in-memory retriever for learning purposes. Production RAG systems typically use a vector database or search backend.


## Install SDK and Required Libraries

In [ ]:
!pip install -q openai numpy

## Add Telnyx API Key

In [ ]:
from getpass import getpass

TELNYX_API_KEY = getpass("Enter your Telnyx API key: ")

## Create Telnyx Client

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=TELNYX_API_KEY,
    base_url="https://api.telnyx.com/v2/ai/openai",
)

## Create a Small Knowledge Base

For this example, we’ll use a few short support documents.

In a real application, these could come from help center articles, product docs, PDFs, tickets, or internal runbooks.

In [ ]:
documents = [
    {
        "title": "API Key Authentication",
        "text": """
Telnyx API requests require a valid API key. If a request returns a 401 error,
check that the API key is active, correctly copied, and has the required permissions.
After rotating an API key, make sure production services are using the new key and
that no old key is cached in environment variables or deployment secrets.
"""
    },
    {
        "title": "Rate Limits",
        "text": """
A 429 error means too many requests were sent in a short period of time.
Applications should use exponential backoff and avoid retry loops that immediately
resend failed requests. If retries are too aggressive, they can increase traffic and
make rate limit errors worse.
"""
    },
    {
        "title": "Webhook Troubleshooting",
        "text": """
Webhook delivery issues can happen when endpoint URLs are incorrect, authentication
headers are missing, or downstream services reject requests. Check request logs,
response codes, and recent configuration changes when debugging webhook failures.
"""
    },
    {
        "title": "Verification Message Delivery",
        "text": """
If users are not receiving verification messages, check message delivery logs,
provider response codes, destination formatting, and whether the signup flow depends
on a backend job that may be failing.
"""
    },
    {
        "title": "Billing Support",
        "text": """
For duplicate charges or invoice questions, collect the account name, invoice IDs,
charge dates, charge amounts, and billing contact. Billing issues are usually not
urgent unless they block account access or service usage.
"""
    },
]

## Generate Embeddings

Embeddings turn text into vectors. Similar pieces of text have similar vectors.

We’ll create one embedding per document using Telnyx’s embedding model.

In [ ]:
# Embedding model: turns documents and questions into vectors for retrieval.
embedding_model = "thenlper/gte-large"

texts = [doc["text"] for doc in documents]

embedding_response = client.embeddings.create(
    model=embedding_model,
    input=texts,
)

for doc, item in zip(documents, embedding_response.data):
    doc["embedding"] = item.embedding

print(f"Created embeddings for {len(documents)} documents.")
print(f"Embedding dimensions: {len(documents[0]['embedding'])}")


## Build a Sample Retriever

For this demo, we’ll use NumPy to compare embeddings in memory.

This keeps the notebook simple and easy to run in Colab. In production, you would usually store and search embeddings with a vector database or search system such as pgvector, ApertureData, Weaviate, Qdrant, Elasticsearch, or another retrieval layer.


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, top_k=3):
    query_embedding = client.embeddings.create(
        model=embedding_model,
        input=query,
    ).data[0].embedding

    scored_docs = []

    for doc in documents:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored_docs.append({
            "title": doc["title"],
            "text": doc["text"],
            "score": score,
        })

    scored_docs = sorted(scored_docs, key=lambda x: x["score"], reverse=True)

    return scored_docs[:top_k]


## Ask a Question

In [ ]:
question = """
Our production signup flow broke after rotating an API key.
Users are not receiving verification codes, and logs show 401 errors.
What should we check first?
"""

retrieved_docs = retrieve(question, top_k=3)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"{i}. {doc['title']} — score: {doc['score']:.3f}")

## Build Context for the Model

In [ ]:
context = "\n\n".join(
    f"Source: {doc['title']}\n{doc['text']}"
    for doc in retrieved_docs
)

print(context)

## Generate a RAG Answer

In [ ]:
# Chat model: generates the final answer using the retrieved context.
chat_model = "moonshotai/Kimi-K2.6"

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful support assistant. "
                "Answer the user's question using only the provided context. "
                "If the context does not contain the answer, say you don't know."
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}

Answer with:
1. A short answer
2. The most relevant sources
"""
        }
    ],
    temperature=0.2,
)

print(response.choices[0].message.content)

## Try Your Own Question

Change the question below and run the next cells again.


In [ ]:
question = "A customer says they were charged twice this month. What information should we collect?"

retrieved_docs = retrieve(question, top_k=3)

context = "\n\n".join(
    f"Source: {doc['title']}\n{doc['text']}"
    for doc in retrieved_docs
)

response = client.chat.completions.create(
    model=chat_model,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful support assistant. "
                "Answer the user's question using only the provided context. "
                "If the context does not contain the answer, say you don't know."
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}

Answer with:
1. A short answer
2. The most relevant sources
"""
        }
    ],
    temperature=0.2,
)

print("Retrieved sources:")
for doc in retrieved_docs:
    print("-", doc["title"])

print("\nAnswer:")
print(response.choices[0].message.content)


## What Just Happened?

You built a simple RAG pipeline:

1. Embedded each document with Telnyx Inference
2. Embedded the user's question
3. Compared the question embedding to each document embedding
4. Retrieved the most relevant documents
5. Passed those documents into a Telnyx chat model as context
6. Generated an answer grounded in the retrieved sources

This example stores vectors in memory. In production, you would usually store embeddings in a vector database or search system.

## Next Steps

Try extending this notebook by:

- Adding more documents
- Splitting long documents into chunks
- Returning more or fewer retrieved sources
- Showing source citations in the final answer
- Storing embeddings in a vector database
- Using your own docs, tickets, or knowledge base content

## Explore More

Want to build more with Telnyx Inference?

- Browse the Telnyx Inference docs: https://developers.telnyx.com/docs/inference
- Learn about Telnyx Inference getting started: https://developers.telnyx.com/docs/inference/getting-started
- Explore Telnyx on GitHub: https://github.com/team-telnyx
- Try the Telnyx Mission Control Portal: https://portal.telnyx.com/